# Koottam: distil the council into a 1B student

Fine-tunes **Gemma 3 1B instruct** with LoRA on `sft.jsonl`, the council's agreed and key-checked answers, and exports GGUF files through one identical pipeline:

| File | What it is |
|---|---|
| `koottam-base.Q8_0.gguf` | the student **before** training (exported with a zero-initialised LoRA, so it equals the base weights). First run only: `EXPORT_BASE` |
| `<STUDENT_NAME>.Q8_0.gguf` | the student **after** training: `koottam-student` in run 1, `koottam-student-balanced` in run 2 |
| `<STUDENT_NAME>.run.json` | every setting and number of the run |

Both go through the same merge, the same converter and the same quantisation, so the only difference between them is the training. That is what makes claim 2 a fair comparison.

**Run it** (free GPU, ~20 min, $0):
- **Colab:** Runtime → Change runtime type → *T4 GPU*, then Runtime → Run all. Upload `data/sft.jsonl` when asked. The GGUFs are copied to Google Drive under `MyDrive/koottam/`.
- **Kaggle:** Settings → Accelerator *GPU T4*, Internet *on*. Add `sft.jsonl` as a private dataset, then Run all. The GGUFs land in `/kaggle/working` (the Output tab).

Then, back on the laptop, follow [Lab 02](../docs/labs/02-distill.md) from step 4.

In [ ]:
%%capture
# Unsloth's own install recipe for Colab/Kaggle (from its Gemma 3 notebook, checked 2026-10-01).
import os
import re

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle gives two T4s; Unsloth trains on one
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)  # noqa: I001
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Settings

Every knob in one place, so the run is recorded in `run.json` and can be repeated.

- **LoRA r=16:** ~2,000 short examples teach a format and a domain, not a new language. A small adapter is enough and overfits less.
- **2 epochs:** one pass is often under-trained on this little data; more starts memorising. The held-out loss printed during training is the check.
- **16-bit, not 4-bit:** a 1B model fits a T4 in 16-bit, and 4-bit training would add a quantisation difference we don't need.
- **Q8_0 GGUF:** small models lose more to 4-bit quantisation than big ones. Q8_0 keeps the comparison about training, not compression.

In [ ]:
BASE_MODEL = "unsloth/gemma-3-1b-it"   # ungated mirror of google/gemma-3-1b-it
CHAT_TEMPLATE = "gemma3"
MAX_SEQ_LENGTH = 1024    # longest example is ~600 tokens
LORA_R, LORA_ALPHA = 16, 32
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH, GRAD_ACCUM = 4, 2  # effective batch 8
HOLDOUT = 0.05            # of sft.jsonl, for the loss curve and a quick sanity check
SEED = 3407
QUANT = "Q8_0"
# Which run this is. Each run adds one student; the base never changes.
#   Run 1 (2026-10-01): "koottam-student" from `build --limit 2500`, 2,094 examples, base exported.
#   Run 2: "koottam-student-balanced" from `build --limit 2500 --all-injection --balance`.
STUDENT_NAME = "koottam-student-balanced"
EXPORT_BASE = False       # True on your first run: the base GGUF is the "before" of claim 2
# The file this run must use, so the wrong sft.jsonl stops the notebook instead of training quietly.
# Run 1 (first run): 2094 examples, sha256 starting 56096d55. Run 2: the values below.
EXPECTED_EXAMPLES = 2298  # `build --limit 2500 --all-injection --balance`, 2026-10-02
EXPECTED_SHA256_PREFIX = "2e660996"

## 1. The training data

The sha256 printed here goes into `run.json`. It ties the trained model to one exact `sft.jsonl`.

In [ ]:
import glob
import hashlib
import json

hits = glob.glob("/kaggle/input/**/sft.jsonl", recursive=True) + glob.glob("sft.jsonl")
if not hits and "COLAB_" in "".join(os.environ.keys()):
    from google.colab import files
    files.upload()            # choose koottam/data/sft.jsonl
    hits = glob.glob("sft.jsonl")
assert hits, "sft.jsonl not found: upload it (Colab) or attach it as a dataset (Kaggle)"
SFT_PATH = hits[0]

raw = open(SFT_PATH, "rb").read()
SFT_SHA256 = hashlib.sha256(raw).hexdigest()
rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
print(f"{SFT_PATH}: {len(rows)} examples, sha256 {SFT_SHA256[:16]}…")
assert len(rows) == EXPECTED_EXAMPLES and SFT_SHA256.startswith(EXPECTED_SHA256_PREFIX), (
    f"WRONG sft.jsonl: got {len(rows)} examples, sha256 {SFT_SHA256[:8]}; "
    f"this run expects {EXPECTED_EXAMPLES}, {EXPECTED_SHA256_PREFIX}. Upload the right file, or edit the settings cell."
)
inj = [r["messages"][-1]["content"].rsplit("ANSWER: ", 1)[1] for r in rows if r["task"] == "injection"]
print(f"injection lessons: {inj.count('A')} safe, {inj.count('B')} attack")
assert all([m["role"] for m in r["messages"]] == ["system", "user", "assistant"] for r in rows)

## 2. Load the model and attach LoRA

In [ ]:
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
)
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)

## 3. Export the *base* student first (first run only)

A fresh LoRA adapter starts with its B matrix at zero, so merging it changes nothing. Exporting now gives the untrained model through the **same** merge → convert → quantise path the trained one will take. The check below proves the adapter really is zero before exporting.

Later runs set `EXPORT_BASE = False`. The base never changes, and the laptop has already scored the file from the first run under the name `koottam-base`. A re-export made with newer library versions could differ, and would then quietly share that name and those cached scores.

In [ ]:
if EXPORT_BASE:
    lora_b = [p for n, p in model.named_parameters() if "lora_B" in n]
    assert lora_b and all(torch.count_nonzero(p) == 0 for p in lora_b), "LoRA is not zero: base export would not be the base"
    print(f"{len(lora_b)} LoRA B matrices, all zero: exporting the untouched base")
    model.save_pretrained_gguf("gguf_base", tokenizer, quantization_method=QUANT)
else:
    print("EXPORT_BASE is off: the laptop keeps the base GGUF from the first run")

## 4. Format the data with the model's chat template

Gemma has no system role: this template folds the system prompt into the first user turn (the system prompt, a blank line, then the question). Ollama's stock Gemma 3 template does **not**: it sends the system prompt as a separate user turn. So `python -m koottam students` serves both students with Koottam's own template, checked to render exactly this text (see `koottam/students.py`).

In [ ]:
from datasets import Dataset


def to_text(batch):
    # Trainer adds <bos> itself; a second one would be a mismatch with serving.
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
                     for m in batch["messages"]]}

ds = Dataset.from_list([{"id": r["id"], "task": r["task"], "messages": r["messages"]} for r in rows])
ds = ds.map(to_text, batched=True)
split = ds.train_test_split(test_size=HOLDOUT, seed=SEED)
train_ds, val_ds = split["train"], split["test"]
print(f"train {len(train_ds)}, held out {len(val_ds)}")
print(train_ds[0]["text"])

## 5. Train on the answers only

`train_on_responses_only` masks the question out of the loss. The student is graded on writing the council's reasoning and answer, not on predicting the question it was just given. The decoded labels below should show only the model's turn.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.05,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="epoch",
        per_device_eval_batch_size=BATCH,
        seed=SEED,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)
labels = trainer.train_dataset[0]["labels"]
print(tokenizer.decode([t for t in labels if t != -100]))

In [ ]:
stats = trainer.train()
history = [h for h in trainer.state.log_history if "eval_loss" in h]
print(f"{stats.metrics['train_runtime'] / 60:.1f} min, final train loss {stats.metrics['train_loss']:.3f}")
for h in history:
    print(f"  epoch {h['epoch']:.0f}: held-out loss {h['eval_loss']:.3f}")

## 6. Sanity check on held-out examples

Not the result: these come from the *training* pool and the real test is the 616 questions scored on the laptop. This only catches a broken run (wrong template, no `ANSWER:` line) before you spend time downloading it.

In [ ]:
ANSWER_RE = re.compile(r"ANSWER\s*[:\-]?\s*\**\s*[\(\[]?\s*([A-Z])\b", re.IGNORECASE)  # same as koottam/prompts.py

def letter(text):
    found = ANSWER_RE.findall(text)
    return found[-1].upper() if found else None

FastModel.for_inference(model)
n, right, parsed = min(40, len(val_ds)), 0, 0
for ex in val_ds.select(range(n)):
    prompt = tokenizer.apply_chat_template(ex["messages"][:2], tokenize=False, add_generation_prompt=True).removeprefix("<bos>")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    got, want = letter(reply), letter(ex["messages"][2]["content"])
    parsed += got is not None
    right += got == want
print(f"held-out sanity: {right}/{n} right, {parsed}/{n} gave a parseable ANSWER line")
print("last reply:\n" + reply)
SANITY = {"n": n, "right": right, "parsed": parsed}

## 7. Export the trained student and collect the files

In [ ]:
import datetime
import importlib.metadata as md
import shutil

model.save_pretrained_gguf("gguf_student", tokenizer, quantization_method=QUANT)

def biggest_gguf(prefix):
    found = glob.glob(f"{prefix}*/**/*.gguf", recursive=True) + glob.glob(f"{prefix}*.gguf")
    assert found, f"no GGUF written under {prefix}*"
    return max(found, key=os.path.getsize)

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "export"
os.makedirs(out_dir, exist_ok=True)
outputs = {}
base = [("gguf_base", "koottam-base")] if EXPORT_BASE else []
for prefix, name in [*base, ("gguf_student", STUDENT_NAME)]:
    dst = os.path.join(out_dir, f"{name}.{QUANT}.gguf")
    shutil.copy(biggest_gguf(prefix), dst)
    outputs[name] = {"file": os.path.basename(dst), "bytes": os.path.getsize(dst)}

run = {
    "when": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"), "student": STUDENT_NAME,
    "base_model": BASE_MODEL, "chat_template": CHAT_TEMPLATE, "quant": QUANT,
    "sft_sha256": SFT_SHA256, "examples": len(rows), "train": len(train_ds), "held_out": len(val_ds),
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "effective_batch": BATCH * GRAD_ACCUM, "seed": SEED, "max_seq_length": MAX_SEQ_LENGTH,
    "train_minutes": round(stats.metrics["train_runtime"] / 60, 1),
    "train_loss": round(stats.metrics["train_loss"], 4),
    "held_out_loss": [round(h["eval_loss"], 4) for h in history],
    "sanity": SANITY, "gpu": torch.cuda.get_device_name(0), "outputs": outputs,
    "versions": {p: md.version(p) for p in ["unsloth", "transformers", "trl", "peft", "torch"]},
}
with open(os.path.join(out_dir, f"{STUDENT_NAME}.run.json"), "w") as f:
    json.dump(run, f, indent=2)
print(json.dumps(run, indent=2))

if "COLAB_" in "".join(os.environ.keys()):
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/koottam"
    os.makedirs(dest, exist_ok=True)
    for f in os.listdir(out_dir):
        shutil.copy(os.path.join(out_dir, f), dest)
    print(f"copied to Google Drive: {dest}")

## Next, on the laptop

Download the new `.gguf` and its `.run.json` (on the first run, also `koottam-base.Q8_0.gguf`) into `koottam/train/outputs/` (gitignored), then:

```bash
python -m koottam students                     # ollama create, for every student file present
python -m koottam eval --model student-base    # first run only
python -m koottam eval --model student-koottam  # run 1; run 2 is student-balanced
python -m koottam compare student-base student-koottam
```

See [Lab 02](../docs/labs/02-distill.md) for what the numbers mean.